# PCA vs LDA vs MDS — 각 기법이 빛나는 순간

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/02_pca_lda_mds_comparison/pca_lda_mds_comparison.ipynb)

## 개요

차원 축소 기법인 **PCA**, **LDA**, **MDS(Isomap)** 는 각자 다른 가정과 목적을 가지고 있습니다.  
이 실습에서는 **데이터의 특성에 따라 어떤 기법이 성공하고 실패하는지**를 직접 눈으로 확인합니다.

| 시나리오 | 성공 기법 | 실패 기법 | 핵심 이유 |
|:--------:|:---------:|:---------:|:---------:|
| A | **LDA** | PCA, MDS | 분산 방향 ≠ 판별 방향 |
| B | **PCA** | LDA, MDS | 다중 모달 클래스 + 고차원 노이즈 |
| C | **MDS (Isomap)** | PCA, LDA | 비선형 매니폴드 구조 |

### 성공/실패 기준
- **구조 보존**: 원본 데이터의 기하학적 구조(클러스터, 매니폴드)가 저차원 투영에서 유지되는가?
- **실루엣 스코어**로 정량화 (1에 가까울수록 클래스 분리 잘 됨)

In [ ]:
# Colab 환경 패키지 설치
import sys
if 'google.colab' in sys.modules:
    !pip install -q plotly scikit-learn

In [ ]:
# ─── 라이브러리 임포트 ───
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import MDS, Isomap
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_swiss_roll
from sklearn.metrics import silhouette_score

print('라이브러리 로드 완료')

In [ ]:
# ─── 하이퍼파라미터 & 전역 설정 ───
RANDOM_STATE   = 42
N_SAMPLES      = 400       # 시나리오 A, B 샘플 수
N_SWISS        = 800       # 시나리오 C (Swiss Roll) 샘플 수

# 시나리오 A — LDA 성공 파라미터
ELONGATION     = 8.0       # x축 방향 표준편차 (길쭉하게)
THIN           = 0.4       # y축 방향 표준편차 (납작하게)
CLASS_SEP_A    = 4.0       # 클래스 간 y축 간격

# 시나리오 B — PCA 성공 파라미터
XOR_DIST       = 6.0       # XOR 클러스터 중심 거리
CLUSTER_STD    = 0.8       # XOR 클러스터 내 분산
N_NOISE_DIMS   = 48        # 추가 노이즈 차원 수
NOISE_STD      = 3.0       # 노이즈 표준편차

# 시나리오 C — MDS 성공 파라미터
SWISS_NOISE    = 0.3       # Swiss Roll 노이즈
ISOMAP_NEIGHBORS = 10      # Isomap k-최근접 이웃

# 시각화 공통
COLORS = ['#e63946', '#457b9d', '#2a9d8f']   # 클래스별 색상
FIG_W, FIG_H = 1100, 380                      # Plotly 캔버스 크기

np.random.seed(RANDOM_STATE)
print('설정 완료')

---
## 시나리오 A — LDA 성공 🏆

### "분산 방향 ≠ 판별 방향" 데이터

**데이터 설계:**  
- 두 클래스 모두 **x축 방향으로 매우 길쭉한** 가우시안 분포  
- 클래스 차이는 **y축 방향**으로만 존재 (위/아래로 분리)

```
  ← Class 1 (y=+4) — 매우 넓게 퍼짐 →
  
  ← Class 2 (y=-4) — 매우 넓게 퍼짐 →
```

| 기법 | 예측 | 이유 |
|------|------|------|
| **PCA** | ✗ 실패 | PC1 = x축(최대 분산) → 두 클래스가 x축에서 완전히 겹침 |
| **LDA** | ✓ 성공 | 판별 방향 = y축 → 완벽하게 분리 |
| **MDS** | ✗ 실패 | 유클리드 거리 → PCA와 유사한 결과 |

In [ ]:
# ─── 시나리오 A: 데이터 생성 ───
n_half = N_SAMPLES // 2

cov_elongated = [[ELONGATION**2, 0],
                 [0,             THIN**2]]

class0_a = np.random.multivariate_normal(
    mean=[0,  CLASS_SEP_A], cov=cov_elongated, size=n_half)
class1_a = np.random.multivariate_normal(
    mean=[0, -CLASS_SEP_A], cov=cov_elongated, size=n_half)

X_a = np.vstack([class0_a, class1_a])
y_a = np.array([0]*n_half + [1]*n_half)

print(f'데이터 shape: {X_a.shape}')
print(f'클래스 0 평균: {class0_a.mean(axis=0).round(2)}')
print(f'클래스 1 평균: {class1_a.mean(axis=0).round(2)}')

In [ ]:
# ─── 시나리오 A: 원본 데이터 시각화 ───
fig_a_orig = go.Figure()
for cls, label, color in zip([0, 1], ['클래스 0 (위)', '클래스 1 (아래)'], COLORS):
    mask = y_a == cls
    fig_a_orig.add_trace(go.Scatter(
        x=X_a[mask, 0], y=X_a[mask, 1],
        mode='markers',
        marker=dict(color=color, size=6, opacity=0.7),
        name=label
    ))

# 축 방향 화살표 주석
fig_a_orig.add_annotation(
    x=14, y=0, ax=0, ay=0,
    xref='x', yref='y', axref='x', ayref='y',
    text='최대 분산 방향 (PC1)',
    showarrow=True, arrowhead=2, arrowwidth=2,
    arrowcolor='gray', font=dict(color='gray', size=11)
)
fig_a_orig.add_annotation(
    x=0, y=6, ax=0, ay=0,
    xref='x', yref='y', axref='x', ayref='y',
    text='판별 방향 (LDA)',
    showarrow=True, arrowhead=2, arrowwidth=2,
    arrowcolor='green', font=dict(color='green', size=11)
)

fig_a_orig.update_layout(
    title='시나리오 A 원본 데이터 — 분산 방향(가로) ≠ 판별 방향(세로)',
    xaxis_title='X (분산 큰 방향)', yaxis_title='Y (판별 방향)',
    width=700, height=420, legend=dict(x=0.01, y=0.99)
)
fig_a_orig.show()

In [ ]:
# ─── 시나리오 A: PCA / LDA / MDS 적용 ───
scaler_a = StandardScaler()
X_a_scaled = scaler_a.fit_transform(X_a)

# PCA
pca_a = PCA(n_components=1)
X_a_pca = pca_a.fit_transform(X_a_scaled)

# LDA (이진 분류: 최대 1개 판별 축)
lda_a = LDA(n_components=1)
X_a_lda = lda_a.fit_transform(X_a_scaled, y_a)

# MDS (고전적 MDS, 유클리드 거리 기반)
mds_a = MDS(n_components=1, metric=True, random_state=RANDOM_STATE, n_init=4)
X_a_mds = mds_a.fit_transform(X_a_scaled)

# 실루엣 스코어 계산
sil_pca_a = silhouette_score(X_a_pca, y_a)
sil_lda_a = silhouette_score(X_a_lda, y_a)
sil_mds_a = silhouette_score(X_a_mds, y_a)

print('=== 시나리오 A 실루엣 스코어 (높을수록 클래스 분리 좋음) ===')
print(f'  PCA : {sil_pca_a:+.4f}  ← ✗ 실패 예상')
print(f'  LDA : {sil_lda_a:+.4f}  ← ✓ 성공 예상')
print(f'  MDS : {sil_mds_a:+.4f}  ← ✗ 실패 예상')

In [ ]:
# ─── 시나리오 A: 비교 시각화 ───
results_a = [
    ('PCA', X_a_pca, sil_pca_a, '✗ 실패', '#e63946'),
    ('LDA', X_a_lda, sil_lda_a, '✓ 성공', '#2a9d8f'),
    ('MDS', X_a_mds, sil_mds_a, '✗ 실패', '#e63946'),
]

fig_a = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f'PCA 투영 (실루엣={sil_pca_a:.3f}) ✗',
        f'LDA 투영 (실루엣={sil_lda_a:.3f}) ✓',
        f'MDS 투영 (실루엣={sil_mds_a:.3f}) ✗',
    ]
)

for col, (method, X_proj, sil, status, border) in enumerate(results_a, 1):
    for cls, color in enumerate(COLORS[:2]):
        mask = y_a == cls
        jitter = np.random.randn(mask.sum()) * 0.05   # 1D → 가독성을 위한 y jitter
        fig_a.add_trace(go.Scatter(
            x=X_proj[mask, 0], y=jitter,
            mode='markers',
            marker=dict(color=color, size=6, opacity=0.7),
            name=f'클래스 {cls}' if col == 1 else None,
            showlegend=(col == 1),
        ), row=1, col=col)

fig_a.update_layout(
    title=dict(text='시나리오 A — LDA 성공: 분산 방향 ≠ 판별 방향', font=dict(size=15)),
    width=FIG_W, height=FIG_H,
    legend=dict(x=0.01, y=-0.15, orientation='h')
)
fig_a.show()

print()
print('해설:')
print('  PCA: x축(최대 분산)으로 투영 → 두 클래스의 x범위가 완전히 겹쳐 분리 불가 ✗')
print('  LDA: y축(판별 방향)으로 투영 → 클래스가 완벽히 분리됨 ✓')
print('  MDS: 유클리드 거리 기반 → PCA와 유사하게 분산이 큰 방향(x)이 지배 ✗')

---
## 시나리오 B — PCA 성공 🏆

### "XOR 다중 모달 클래스 + 고차원 노이즈" 데이터

**데이터 설계:**  
- 4개 클러스터가 XOR 패턴으로 배치 (2개 클래스 각각 2개 모드)
- 2D 구조에 **48개 노이즈 차원** 추가 → 총 50차원

```
  Class 0 ●    Class 1 ●
    (+6,+6)      (+6,-6)
    
  Class 1 ●    Class 0 ●
    (-6,+6)      (-6,-6)
```

| 기법 | 예측 | 이유 |
|------|------|------|
| **PCA** | ✓ 성공 | 유의미한 2D 방향(분산 큰)이 노이즈 차원 압도 → 4 클러스터 구조 복원 |
| **LDA** | ✗ 실패 | 두 클래스의 평균이 모두 원점 (μ₀=μ₁=(0,0)) → 판별 방향 없음 |
| **MDS** | ✗ 실패 | 50차원에서 노이즈가 거리를 지배 → 클러스터 구조 소실 |

In [ ]:
# ─── 시나리오 B: 데이터 생성 ───
n_quarter = N_SAMPLES // 4
D = XOR_DIST

# XOR 패턴: 클래스 0 → 1·4사분면, 클래스 1 → 2·3사분면
centers = np.array([[+D, +D], [-D, -D],   # Class 0 클러스터 중심
                    [+D, -D], [-D, +D]])   # Class 1 클러스터 중심
labels_b_base = [0, 0, 1, 1]

X_b_2d_parts = [
    np.random.multivariate_normal(
        mean=centers[i],
        cov=np.eye(2) * CLUSTER_STD**2,
        size=n_quarter
    )
    for i in range(4)
]
X_b_2d = np.vstack(X_b_2d_parts)
y_b     = np.array([lbl for lbl, _ in zip(labels_b_base, range(4))
                    for _ in range(n_quarter)])

# 두 클래스의 평균 확인 (이론상 동일)
mean_c0 = X_b_2d[y_b == 0].mean(axis=0)
mean_c1 = X_b_2d[y_b == 1].mean(axis=0)
print(f'클래스 0 평균 (2D): {mean_c0.round(3)}')
print(f'클래스 1 평균 (2D): {mean_c1.round(3)}')
print(f'클래스 간 평균 차이: {np.linalg.norm(mean_c0 - mean_c1):.4f}  ← 거의 0 → LDA 실패 원인')

# 48개 노이즈 차원 추가 → 50D
noise_b = np.random.randn(N_SAMPLES, N_NOISE_DIMS) * NOISE_STD
X_b     = np.hstack([X_b_2d, noise_b])
print(f'\n최종 데이터 shape: {X_b.shape}  (유효 2D + 노이즈 {N_NOISE_DIMS}D)')

In [ ]:
# ─── 시나리오 B: 원본 2D 구조 시각화 ───
fig_b_orig = go.Figure()
for cls, color in zip([0, 1], COLORS[:2]):
    mask = y_b == cls
    fig_b_orig.add_trace(go.Scatter(
        x=X_b_2d[mask, 0], y=X_b_2d[mask, 1],
        mode='markers',
        marker=dict(color=color, size=7, opacity=0.7),
        name=f'클래스 {cls}'
    ))

# 클래스 평균 표시
for cls, color, mean in zip([0, 1], COLORS[:2], [mean_c0, mean_c1]):
    fig_b_orig.add_trace(go.Scatter(
        x=[mean[0]], y=[mean[1]],
        mode='markers+text',
        marker=dict(color=color, size=18, symbol='star'),
        text=[f'μ{cls}≈(0,0)'], textposition='top center',
        showlegend=False
    ))

fig_b_orig.update_layout(
    title='시나리오 B 원본 구조 (2D 시각화) — 두 클래스 평균이 모두 원점에 가까움',
    xaxis_title='X', yaxis_title='Y',
    width=600, height=420
)
fig_b_orig.show()

In [ ]:
# ─── 시나리오 B: PCA / LDA / MDS 적용 (50D 입력) ───
scaler_b = StandardScaler()
X_b_scaled = scaler_b.fit_transform(X_b)

# PCA
pca_b = PCA(n_components=2)
X_b_pca = pca_b.fit_transform(X_b_scaled)

# LDA (이진 분류 → 최대 1D)
lda_b = LDA(n_components=1)
X_b_lda = lda_b.fit_transform(X_b_scaled, y_b)

# MDS (50D 유클리드 거리 기반)
mds_b = MDS(n_components=2, metric=True, random_state=RANDOM_STATE, n_init=4)
X_b_mds = mds_b.fit_transform(X_b_scaled)

# 실루엣 스코어
sil_pca_b = silhouette_score(X_b_pca, y_b)
sil_lda_b = silhouette_score(X_b_lda, y_b)
sil_mds_b = silhouette_score(X_b_mds, y_b)

# LDA 판별 방향의 크기 확인
between_class_dist = np.linalg.norm(lda_b.means_[0] - lda_b.means_[1])

print('=== 시나리오 B 실루엣 스코어 (높을수록 클래스 분리 좋음) ===')
print(f'  PCA : {sil_pca_b:+.4f}  ← ✓ 성공 예상')
print(f'  LDA : {sil_lda_b:+.4f}  ← ✗ 실패 예상')
print(f'  MDS : {sil_mds_b:+.4f}  ← ✗ 실패 예상')
print(f'\nLDA 스케일된 공간에서 클래스 간 거리: {between_class_dist:.4f}  ← 거의 0 → LDA 실패')

In [ ]:
# ─── 시나리오 B: 비교 시각화 ───
def make_scatter_subplot(fig, X_proj, y, colors, row, col,
                         is_1d=False, show_legend=False):
    """서브플롯에 산점도 추가 (1D/2D 공통 처리)"""
    for cls, color in enumerate(colors):
        mask = y == cls
        x_vals = X_proj[mask, 0]
        if is_1d:
            y_vals = np.random.randn(mask.sum()) * 0.1
        else:
            y_vals = X_proj[mask, 1]
        fig.add_trace(go.Scatter(
            x=x_vals, y=y_vals,
            mode='markers',
            marker=dict(color=color, size=6, opacity=0.7),
            name=f'클래스 {cls}' if show_legend else None,
            showlegend=show_legend,
            legendgroup=f'cls{cls}',
        ), row=row, col=col)

fig_b = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f'PCA (실루엣={sil_pca_b:.3f}) ✓',
        f'LDA (실루엣={sil_lda_b:.3f}) ✗',
        f'MDS (실루엣={sil_mds_b:.3f}) ✗',
    ]
)

make_scatter_subplot(fig_b, X_b_pca, y_b, COLORS, 1, 1, show_legend=True)
make_scatter_subplot(fig_b, X_b_lda, y_b, COLORS, 1, 2, is_1d=True)
make_scatter_subplot(fig_b, X_b_mds, y_b, COLORS, 1, 3)

fig_b.update_layout(
    title=dict(text='시나리오 B — PCA 성공: XOR 다중 모달 + 고차원 노이즈', font=dict(size=15)),
    width=FIG_W, height=FIG_H,
    legend=dict(x=0.01, y=-0.15, orientation='h')
)
fig_b.show()

print()
print('해설:')
print('  PCA: 노이즈 차원보다 유효 차원(XOR 2D)의 분산이 커 → 4 클러스터 구조 복원 ✓')
print('  LDA: 클래스 0, 1의 평균이 모두 원점 → 판별 방향 계산 불가 ✗')
print('  MDS: 50D에서 노이즈가 거리 지배 → XOR 구조 사라짐 ✗')

---
## 시나리오 C — MDS (Isomap) 성공 🏆

### "Swiss Roll 비선형 매니폴드" 데이터

**데이터 설계:**  
- sklearn의 `make_swiss_roll`로 생성한 3D 데이터  
- 실제 구조: 2D 평면이 롤케이크처럼 말려있는 매니폴드  
- 성공 = "롤을 펼쳐서" 원래 2D 평면 구조 복원

```
원본 (말린 상태)     PCA 투영 (선형)     Isomap (측지선 거리)
   🌀 Swiss Roll  →  📎 접힌 형태   /   📄 펼친 평면
```

| 기법 | 예측 | 이유 |
|------|------|------|
| **PCA** | ✗ 실패 | 선형 투영 → 말린 층이 겹쳐 매니폴드 구조 파괴 |
| **LDA** | ✗ 실패 | 선형 판별 → 비선형 구조 인식 불가 |
| **MDS (Isomap)** | ✓ 성공 | 측지선(표면을 따른 거리) → 매니폴드 성공적으로 펼침 |

In [ ]:
# ─── 시나리오 C: Swiss Roll 생성 ───
X_c, t_c = make_swiss_roll(n_samples=N_SWISS, noise=SWISS_NOISE, random_state=RANDOM_STATE)

# t_c: 롤의 각도 위치 (연속값) → 구조 보존 여부 색상으로 표현
# LDA용 클래스: 4등분
t_bins = np.percentile(t_c, [25, 50, 75])
y_c = np.digitize(t_c, bins=t_bins)  # 0,1,2,3 → 4 클래스

print(f'Swiss Roll shape: {X_c.shape}')
print(f'매니폴드 파라미터 t 범위: {t_c.min():.2f} ~ {t_c.max():.2f}')
print(f'클래스 분포: {np.bincount(y_c)}')

In [ ]:
# ─── 시나리오 C: 3D 원본 시각화 ───
fig_c_3d = go.Figure(data=[go.Scatter3d(
    x=X_c[:, 0], y=X_c[:, 1], z=X_c[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=t_c,
        colorscale='Rainbow',
        opacity=0.8,
        colorbar=dict(title='t (매니폴드 위치)')
    )
)])

fig_c_3d.update_layout(
    title='시나리오 C 원본 — Swiss Roll (3D 매니폴드)',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z'
    ),
    width=700, height=520
)
fig_c_3d.show()
print('색상이 연속적으로 변해야 구조가 잘 보존된 것 (매니폴드 위치를 색상으로 표현)')

In [ ]:
# ─── 시나리오 C: PCA / LDA / Isomap(MDS) 적용 ───
scaler_c = StandardScaler()
X_c_scaled = scaler_c.fit_transform(X_c)

# PCA
pca_c = PCA(n_components=2)
X_c_pca = pca_c.fit_transform(X_c_scaled)

# LDA (4클래스 → 최대 3D, 여기서 2D 사용)
lda_c = LDA(n_components=2)
X_c_lda = lda_c.fit_transform(X_c_scaled, y_c)

# Isomap (측지선 거리 기반 MDS)
isomap_c = Isomap(n_neighbors=ISOMAP_NEIGHBORS, n_components=2)
X_c_isomap = isomap_c.fit_transform(X_c_scaled)

# 실루엣 스코어 (4클래스 기준)
sil_pca_c    = silhouette_score(X_c_pca,    y_c)
sil_lda_c    = silhouette_score(X_c_lda,    y_c)
sil_isomap_c = silhouette_score(X_c_isomap, y_c)

print('=== 시나리오 C 실루엣 스코어 ===')
print(f'  PCA    : {sil_pca_c:+.4f}  ← ✗ 실패 예상')
print(f'  LDA    : {sil_lda_c:+.4f}  ← ✗ 실패 예상')
print(f'  Isomap : {sil_isomap_c:+.4f}  ← ✓ 성공 예상')

In [ ]:
# ─── 시나리오 C: 비교 시각화 (색상 = 매니폴드 위치 t) ───
def swiss_scatter(X_proj, t_vals, title):
    """매니폴드 위치(t)를 색상으로 표현한 2D 산점도 반환"""
    return go.Scatter(
        x=X_proj[:, 0], y=X_proj[:, 1],
        mode='markers',
        marker=dict(
            color=t_vals, colorscale='Rainbow',
            size=4, opacity=0.8,
            showscale=False
        ),
        name=title, showlegend=False
    )

fig_c = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f'PCA (실루엣={sil_pca_c:.3f}) ✗',
        f'LDA (실루엣={sil_lda_c:.3f}) ✗',
        f'Isomap-MDS (실루엣={sil_isomap_c:.3f}) ✓',
    ]
)

fig_c.add_trace(swiss_scatter(X_c_pca,    t_c, 'PCA'),    row=1, col=1)
fig_c.add_trace(swiss_scatter(X_c_lda,    t_c, 'LDA'),    row=1, col=2)
fig_c.add_trace(swiss_scatter(X_c_isomap, t_c, 'Isomap'), row=1, col=3)

fig_c.update_layout(
    title=dict(text='시나리오 C — MDS(Isomap) 성공: Swiss Roll 매니폴드', font=dict(size=15)),
    width=FIG_W, height=FIG_H + 30
)
fig_c.show()

print()
print('해설 (색상이 무지개처럼 연속적으로 변하면 매니폴드 구조 잘 보존):')
print('  PCA    : 색상이 섞여있음 → 롤이 접혀 인접 층이 겹침 ✗')
print('  LDA    : 4등분 클래스 기준 → 롤 구조 무시, 클래스 경계만 추구 ✗')
print('  Isomap : 색상이 연속적으로 변함 → 롤이 펼쳐져 매니폴드 구조 복원 ✓')

---
## 종합 비교

In [ ]:
# ─── 종합 실루엣 스코어 비교 ───
import pandas as pd

summary = pd.DataFrame({
    '기법': ['PCA', 'LDA', 'MDS / Isomap'],
    'A: 길쭉한 가우시안': [sil_pca_a, sil_lda_a, sil_mds_a],
    'B: XOR + 고차원노이즈': [sil_pca_b, sil_lda_b, sil_mds_b],
    'C: Swiss Roll': [sil_pca_c, sil_lda_c, sil_isomap_c],
})
summary = summary.set_index('기법')

print('=== 실루엣 스코어 요약 (★: 해당 시나리오 최고점) ===')
print(summary.round(4).to_string())

# 각 시나리오 최고 기법 표시
for col in summary.columns:
    winner = summary[col].idxmax()
    print(f'  {col} → 최고 기법: {winner} ({summary.loc[winner, col]:.4f})')

In [ ]:
# ─── 종합 비교 막대 차트 ───
methods  = ['PCA', 'LDA', 'MDS/Isomap']
scores_a = [sil_pca_a, sil_lda_a, sil_mds_a]
scores_b = [sil_pca_b, sil_lda_b, sil_mds_b]
scores_c = [sil_pca_c, sil_lda_c, sil_isomap_c]

scenario_colors = ['#457b9d', '#e9c46a', '#e76f51']

fig_summary = go.Figure()
for scores, name, color in zip(
    [scores_a, scores_b, scores_c],
    ['A: 길쭉한 가우시안 (LDA 성공)', 'B: XOR+고차원노이즈 (PCA 성공)', 'C: Swiss Roll (Isomap 성공)'],
    scenario_colors
):
    fig_summary.add_trace(go.Bar(
        name=name,
        x=methods,
        y=scores,
        marker_color=color,
        opacity=0.85
    ))

fig_summary.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.5)
fig_summary.update_layout(
    title='종합 비교 — 시나리오별 실루엣 스코어 (높을수록 구조 보존 우수)',
    barmode='group',
    yaxis_title='실루엣 스코어',
    xaxis_title='기법',
    width=800, height=500,
    legend=dict(x=0, y=1.15, orientation='h')
)
fig_summary.show()

---
## 결론 및 기법 선택 가이드

| 기법 | 언제 쓰나 | 가정 | 실패 조건 |
|------|-----------|------|----------|
| **PCA** | 레이블 없음, 전역 분산 구조 파악 | 선형, 분산 큰 방향 = 의미있는 방향 | 비선형 구조, 분산 방향 ≠ 의미 |
| **LDA** | 레이블 있음, 클래스 판별 | 선형, 클래스 간 평균 다름, 가우시안 | 클래스 평균 동일, 다중 모달, 비선형 |
| **MDS/Isomap** | 거리·유사도 기반 구조 복원 | 거리 행렬이 의미있음 | 고차원 노이즈, 유클리드 거리 부적합 |

### 핵심 교훈
> **"모든 상황에서 최고인 차원 축소 기법은 없다."**  
> 데이터의 특성(레이블 유무, 선형/비선형, 차원 수)을 먼저 파악하고, 목적에 맞는 기법을 선택해야 한다.